In [ ]:
from transformers.utils import logging
logging.set_verbosity_error()

In [ ]:
from dotenv import load_dotenv
from huggingface_hub import login, whoami

load_dotenv()
login()
whoami()['name']

In [ ]:
from transformers import pipeline

classifier = pipeline("text-classification")
classifier("This course is amazing")

In [ ]:
tokenizer = classifier.tokenizer
model = classifier.model

print(type(tokenizer))
print(type(model))

In [ ]:
print(model.config.id2label)

In [ ]:
text = "This course is amazing"

inputs = tokenizer(text, return_tensors="pt")
print(inputs)
print(f"Input IDs: {inputs['input_ids']}")
print(f"Attention Mask: {inputs['attention_mask']}")

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
tokens

In [ ]:
outputs = model(**inputs, output_hidden_states=True)
print(outputs)
print(f"Predictions (logits): {outputs['logits']}")

In [ ]:
embeddings = model.distilbert.embeddings
word_embeddings = embeddings.word_embeddings(inputs['input_ids'])
print(word_embeddings.shape)
print(word_embeddings)

In [ ]:
import torch

seq_length = inputs['input_ids'].shape[1]
position_ids = torch.arange(seq_length).unsqueeze(0)
position_embeddings = embeddings.position_embeddings(position_ids)
print(position_embeddings.shape)
print(position_embeddings)

In [ ]:
hidden_states = outputs.hidden_states
print(len(hidden_states))
for i, hidden in enumerate(hidden_states):
    print(i, hidden.shape)

In [ ]:
token_index = tokens.index("amazing")

for layer_idx, hidden in enumerate(hidden_states):
    vector = hidden[0, token_index]
    print(f"Layer {layer_idx}:", vector[:5])

In [ ]:
import torch

probs = torch.softmax(outputs.logits, dim=-1)
probs

In [ ]:
predicted_id = torch.argmax(probs, dim=-1).item()

label = model.config.id2label[predicted_id]
score = probs[0][predicted_id].item()

print(f"Predicted Label: {label}, Score: {score}")

In [ ]:
classifier(text)

**Excercise 2.1**

text-classification 태스크에서 다음 모델을 적용한 후에 

* nlptown/bert-base-multilingual-uncased-sentiment 

다음 문장의 분류 결과를 확인해보세요.

* 난 널 사랑해.

In [ ]:
from transformers import pipeline

text = "난 널 사랑해."

classifier = pipeline(
    task="text-classification", 
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)
classifier(text)

**Exercise 2.2**

위 1번 문제의 파이프라인에서 

이 파이프라인의 Tokenizer와 Model의 파이썬 클래스 이름을 확인하세요.

이 모델이 분류하는 분류 클래스 라벨의 리스트(model.config.id2label)를 출력해보세요.

이 모델의 토크나이저가 생성한 토큰 아이디들을 토큰으로 변환해 출력해보세요.

이 모델의 Model Head에 있는 logits의 차원(shape)과 값(logits)을 확인해보세요.

In [ ]:
tokenizer = classifier.tokenizer
model = classifier.model
print(type(tokenizer))
print(type(model))
print(model.config.id2label)

In [ ]:
inputs = tokenizer(text, return_tensors="pt")
print(*tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))

In [ ]:
outputs = model(**inputs)
print(f"shape: {outputs.logits.shape}")
print(f"logits: {outputs.logits}")

**Exercise 2.3**

위 2번 문제의 logits를 직접 softmax 함수로 확률값으로 변환하세요.

위에서 계산한 확률값을 라벨과 점수를 선택하고, 파이프라인의 결과값과 비교해보세요.


In [ ]:
import torch

probs = torch.softmax(outputs.logits, dim=-1)
print(probs)
pred_id = torch.argmax(probs, dim=-1).item()
label = model.config.id2label[pred_id]
score = probs[0][pred_id].item()
print(f"Label: {label}, Score: {score}")

In [ ]:
classifier(text)

In [ ]:
from transformers import pipeline

text = "How are you?"
generator = pipeline("text-generation")
generator(text, max_new_tokens=10)

In [ ]:
tokenizer = generator.tokenizer
model = generator.model
print(type(tokenizer))
print(type(model))

In [ ]:
inputs = tokenizer(text, return_tensors="pt")
print(f"Input IDs: {inputs['input_ids']}")
print(f"Attention Mask: {inputs['attention_mask']}")
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
print(f"Tokens: {tokens}")

In [ ]:
outputs = model(**inputs, output_hidden_states=True)
print(f"Shape: {outputs.logits.shape}")
print(f"Predictions (logits): {outputs.logits}")

In [ ]:
hidden_states = outputs.hidden_states
print(len(hidden_states))
print(hidden_states[-1].shape)
print(hidden_states[-1])

In [ ]:
import torch

# 마지막 위치에서 다음에 올 token 후보에 대한 점수(확률) 계산
next_token_logits = outputs.logits[:, -1, :] 
probs = torch.softmax(next_token_logits, dim=-1)
topk_probs, topk_token_ids = torch.topk(probs, k=3, dim=-1)
for rank, (prob, token_id) in enumerate(zip(topk_probs[0], topk_token_ids[0]), start=1):
    token = tokenizer.decode(token_id)
    print(f"Rank: {rank}, Token:{token}, Probability: {prob:.4f}")